# 29 · Multimodal RAG

**Idea:** text embeddings can't search images directly. The pragmatic pattern: use a vision model to **caption** each image, index the captions as text, and retrieve normally — a query matches the caption, and you return the image it describes. This reuses everything from the earlier notebooks; only the ingestion step changes.

> Caption-based retrieval runs on Ollama embeddings. Actual image captioning needs a vision model (Gemini/Anthropic) — that cell skips if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Start with captions already written (as if a vision model produced them). `MultimodalStore` embeds the captions and retrieves images by text similarity — no image bytes needed to see the retrieval mechanics.

In [ ]:
from rag_lab import MultimodalStore, ImageDoc

images = [
    ImageDoc('img1', 'A line chart showing retrieval recall increasing as more clusters are probed in an IVF index.'),
    ImageDoc('img2', 'A diagram of a RAG pipeline: query embedding, vector search, reranking, then an LLM generating an answer.'),
    ImageDoc('img3', 'A photo of a golden retriever puppy sitting in a grassy park.'),
    ImageDoc('img4', 'A bar chart comparing BM25 keyword search against dense vector search across several queries.'),
]
mm = MultimodalStore(provider='ollama'); mm.add(images)
print(f'indexed {len(mm)} images')

Query in plain text — the store returns the images whose captions best match.

In [ ]:
for query in ['how does the RAG pipeline flow?', 'keyword vs vector comparison', 'a cute dog']:
    top, score = mm.search(query, k=1)[0]
    print(f'{query!r:45} -> {top.id} ({score:.3f})  {top.caption[:60]}...')

To ingest *real* images, `caption_image` calls a vision model. Drop a PNG/JPG path in and it captions, then indexes it like any other doc. Skips if no vision key or no sample image is present.

In [ ]:
from pathlib import Path
from rag_lab import caption_image

sample = next(iter(Path.cwd().rglob('*.png')), None)
try:
    if sample is None:
        raise FileNotFoundError('no sample .png found in the workspace')
    caption = caption_image(sample, provider='gemini')
    print(f'{sample.name} -> {caption}')
    mm.add([ImageDoc(sample.name, caption, path=str(sample))])
    print('indexed; store now holds', len(mm), 'images')
except Exception as e:
    print('captioning skipped:', type(e).__name__, e)

**Takeaway:** multimodal RAG doesn't require multimodal *embeddings* — captioning turns images into text, so the entire retrieval stack (embed → search → rerank → cite) works unchanged. The same trick extends to audio (transcripts) and tables (serialized rows).

That completes the expert block (18–29). Next → `capstone.ipynb`, which assembles hybrid retrieval + reranking + compression + citations behind an evaluation harness and a live cost meter.